In [1]:
from dopplerguesser.predict.fetch_tles import fetch_tles
from dopplerguesser.predict.filters import filter_visibility, filter_heo, filter_geostationary, filter_by_doppler, filter_constellations
from dopplerguesser.predict.matcher import score_candidates
from dopplerguesser.predict.observer import Observer
from dopplerguesser.predict.satellite import Satellite
from dopplerguesser.predict.propagator import propagate_earth_rotation, propagate_fg_elliptic

In [2]:
from time import perf_counter
from datetime import datetime
from datetime import timezone

In [3]:
observer = Observer(lat=55.7558, lon=37.6176, alt=200)  # Moscow

In [4]:
now = datetime.now(tz=timezone.utc)
print(now)
ts = now.timestamp()
print(ts)

2026-01-28 20:51:10.990350+00:00
1769633470.99035


In [5]:
fetched_tles = fetch_tles(ts)

In [6]:
with open('activesatellites.txt', 'w') as f:
    for sat in fetched_tles:
        f.write(f"{sat.satellite.name}\n")

In [7]:
# Filtering
print('Initial length:', len(fetched_tles))

t1 = perf_counter()
satellites = filter_constellations(fetched_tles, constellations_to_remove=['starlink', 'oneweb'])
t2 = perf_counter()
print('After constellation filter:', len(satellites))
print('Took {:.3f} seconds\n'.format(t2 - t1))

t1 = perf_counter()
satellites = filter_geostationary(satellites)
t2 = perf_counter()
print('After geostationary filter:', len(satellites))
print('Took {:.3f} seconds\n'.format(t2 - t1))

t1 = perf_counter()
satellites = filter_heo(satellites)
t2 = perf_counter()
print('After HEO filter:', len(satellites))
print('Took: {:.3f} seconds\n'.format(t2 - t1))

t1 = perf_counter()
satellites = filter_visibility(satellites, observer, ts, min_elevation=10.0)
t2 = perf_counter()
print('After visibility filter:', len(satellites))
print('Took: {:.3f} seconds\n'.format(t2 - t1))

Initial length: 14301
After constellation filter: 4106
Took 0.009 seconds

After geostationary filter: 3305
Took 0.002 seconds

After HEO filter: 3303
Took: 0.001 seconds

After visibility filter: 82
Took: 0.230 seconds



In [16]:
# Propagation
t1 = perf_counter()
observer.compute_track(ts)
t2 = perf_counter()
print('Earth rotation propagation took {:.3f} seconds\n'.format(t2 - t1))

Earth rotation propagation took 0.014 seconds

